# Fetching papers from arXiv



## Installing what we need

Two packages. `arxiv` is the client. `tqdm` just gives us a progress bar so we're not staring at a blank cell wondering if anything is happening.

In [ ]:
! uv pip install arxiv tqdm pymupdf

In [30]:
import arxiv
import fitz  # PyMuPDF
import json
import tempfile
from pathlib import Path
from tqdm import tqdm

## Where things go



In [31]:
DATA_DIR = Path("./papers")
TXT_DIR = DATA_DIR / "texts"
META_DIR = DATA_DIR / "metadata"

TXT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print(TXT_DIR.resolve())
print(META_DIR.resolve())

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/arxivx/papers/texts
/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/arxivx/papers/metadata


## Writing the query

arXiv's query syntax lets you scope searches to the title, abstract, author, or category. You combine terms with `AND`, `OR`, `ANDNOT`. So `abs:"retrieval augmented generation"` means the phrase retrieval augmented generation appears in the abstract, and `cat:cs.CL` restricts to the computation-and-language category.

I'll search for recent RAG papers, which is a bit of a joke on ourselves but it's also useful material for the collection we're building.

In [41]:
QUERY = 'abs:"spiking neural network" AND cat:cs.CL'
MAX_RESULTS = 100

search = arxiv.Search(
    query=QUERY,
    max_results=MAX_RESULTS,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)

# arXiv asks clients to wait a few seconds between requests. The Client
# object takes care of that (and of retries) so we don't have to.
client = arxiv.Client(page_size=10, delay_seconds=3, num_retries=3)


## Looking before we download



In [42]:
results = list(client.results(search))

print(f"Got {len(results)} papers.\n")

for i, p in enumerate(results, 1):
    authors = ', '.join(a.name for a in p.authors[:3])
    if len(p.authors) > 3:
        authors += ', ...'
    print(f"{i}. {p.title}")
    print(f"   {authors}")
    print(f"   {p.published.date()}   {p.get_short_id()}")
    print()

Got 21 papers.

1. MAR: Efficient Large Language Models via Module-aware Architecture Refinement
   Junhong Cai, Guiqin Wang, Kejie Zhao, ...
   2026-01-29   2601.21503v1

2. Efficient Aspect Term Extraction using Spiking Neural Network
   Abhishek Kumar Mishra, Arya Somasundaram, Anup Das, ...
   2026-01-10   2601.06637v1

3. BrainTransformers: SNN-LLM
   Zhengzheng Tang, Eva Zhu
   2024-10-03   2410.14687v2

4. Bio-Inspired Mamba: Temporal Locality and Bioplausible Learning in Selective State Space Models
   Jiahao Qin
   2024-09-17   2409.11263v1

5. Sorbet: A Neuromorphic Hardware-Compatible Transformer-Based Spiking Language Model
   Kaiwen Tang, Zhanglu Yan, Weng-Fai Wong
   2024-09-04   2409.15298v2

6. SpikingSSMs: Learning Long Sequences with Sparse and Parallel Spiking State Space Models
   Shuaijie Shen, Chao Wang, Renzhuo Huang, ...
   2024-08-27   2408.14909v2

7. Spiking Convolutional Neural Networks for Text Classification
   Changze Lv, Jianhan Xu, Xiaoqing Zheng
   202

Each paper object has a bunch of attributes beyond what I printed. The interesting ones are `title`, `authors`, `summary` (which is the abstract, confusingly), `categories`, `published`, `pdf_url`, and `entry_id`. We'll save most of those.

## One quick helper for IDs



In [34]:
def paper_id(paper):
    return paper.get_short_id().split('v')[0]

paper_id(results[0])

'2601.21503'

## Downloading one paper by hand

Before we loop, let's do one the slow way. I want you to see what actually happens on each step.

In [35]:
paper = results[0]
pid = paper_id(paper)

txt_path = TXT_DIR / f"{pid}.txt"
meta_path = META_DIR / f"{pid}.json"

with tempfile.TemporaryDirectory() as tmp:
    paper.download_pdf(dirpath=tmp, filename=f"{pid}.pdf")
    doc = fitz.open(f"{tmp}/{pid}.pdf")
    text = "\n".join(page.get_text() for page in doc)
    doc.close()

txt_path.write_text(text, encoding="utf-8")
print(f"{txt_path}  ({len(text):,} chars)")

papers/texts/2601.21503.txt  (25,292 chars)


In [36]:
meta = {
    "arxiv_id": pid,
    "title": paper.title,
    "authors": [a.name for a in paper.authors],
    "abstract": paper.summary,
    "categories": paper.categories,
    "primary_category": paper.primary_category,
    "published": paper.published.isoformat(),
    "updated": paper.updated.isoformat(),
    "pdf_url": paper.pdf_url,
    "entry_id": paper.entry_id,
    "doi": paper.doi,
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

# Show everything except the abstract, which is long
print(json.dumps({k: v for k, v in meta.items() if k != 'abstract'}, indent=2))

{
  "arxiv_id": "2601.21503",
  "title": "MAR: Efficient Large Language Models via Module-aware Architecture Refinement",
  "authors": [
    "Junhong Cai",
    "Guiqin Wang",
    "Kejie Zhao",
    "Jianxiong Tang",
    "Xiang Wang",
    "Luziwei Leng",
    "Ran Cheng",
    "Yuxin Ma",
    "Qinghai Guo"
  ],
  "categories": [
    "cs.AI",
    "cs.CL",
    "cs.LG",
    "cs.NE"
  ],
  "primary_category": "cs.AI",
  "published": "2026-01-29T10:21:28+00:00",
  "updated": "2026-01-29T10:21:28+00:00",
  "pdf_url": "https://arxiv.org/pdf/2601.21503v1",
  "entry_id": "http://arxiv.org/abs/2601.21503v1",
  "doi": null
}


Open the JSON file in a text editor if you want. It's human-readable and will make sense.

## Now the loop



In [38]:
def download(paper, txt_dir, meta_dir, overwrite=False):
    pid = paper_id(paper)
    txt_path = txt_dir / f"{pid}.txt"
    meta_path = meta_dir / f"{pid}.json"

    if not overwrite and txt_path.exists() and meta_path.exists():
        return pid, "skip"

    try:
        with tempfile.TemporaryDirectory() as tmp:
            paper.download_pdf(dirpath=tmp, filename=f"{pid}.pdf")
            doc = fitz.open(f"{tmp}/{pid}.pdf")
            text = "\n".join(page.get_text() for page in doc)
            doc.close()
        txt_path.write_text(text, encoding="utf-8")

        meta = {
            "arxiv_id": pid,
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "abstract": paper.summary,
            "categories": paper.categories,
            "primary_category": paper.primary_category,
            "published": paper.published.isoformat(),
            "updated": paper.updated.isoformat(),
            "pdf_url": paper.pdf_url,
            "entry_id": paper.entry_id,
            "doi": paper.doi,
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

        return pid, "ok"
    except Exception as e:
        return pid, f"error: {e}"

In [43]:
for paper in tqdm(results):
    pid, status = download(paper, TXT_DIR, META_DIR)
    tqdm.write(f"{status:>6}  {pid}")

  0%|          | 0/21 [00:00<?, ?it/s]

  skip  2601.21503
  skip  2601.06637
  skip  2410.14687
  skip  2409.11263
  skip  2409.15298


 29%|██▊       | 6/21 [00:01<00:03,  4.74it/s]

    ok  2408.14909


 33%|███▎      | 7/21 [00:03<00:08,  1.63it/s]

...

    ok  2011.06846


 95%|█████████▌| 20/21 [00:36<00:01,  1.69s/it]

    ok  1911.08373


100%|██████████| 21/21 [00:37<00:00,  1.78s/it]

    ok  1905.11235


## Checking what actually landed

Don't trust the log output. Look at the disk.

In [44]:
txts = sorted(TXT_DIR.glob("*.txt"))
metas = sorted(META_DIR.glob("*.json"))

total_kb = sum(p.stat().st_size for p in txts) / 1024

print(f"{len(txts)} text files, {len(metas)} metadata files, {total_kb:.0f} KB total\n")

for p in txts:
    print(f"  {p.name}  ({p.stat().st_size / 1024:.0f} KB)")

26 text files, 26 metadata files, 1447 KB total

  1905.11235.txt  (31 KB)
  1911.08373.txt  (69 KB)
  2011.06846.txt  (28 KB)
  2212.01187.txt  (24 KB)
  2302.13939.txt  (84 KB)
  2308.09455.txt  (59 KB)
  2308.15122.txt  (59 KB)
  2310.06488.txt  (77 KB)
  2312.09084.txt  (28 KB)
  2401.06808.txt  (37 KB)
  2401.17911.txt  (50 KB)
  2403.18609.txt  (70 KB)
  2404.14024.txt  (87 KB)
  2406.03287.txt  (65 KB)
  2406.19230.txt  (66 KB)
  2408.14909.txt  (66 KB)
  2409.11263.txt  (33 KB)
  2409.15298.txt  (56 KB)
  2410.14687.txt  (45 KB)
  2601.06637.txt  (70 KB)
  2601.21503.txt  (25 KB)
  2604.18105.txt  (71 KB)
  2604.18257.txt  (61 KB)
  2604.18362.txt  (85 KB)
  2604.18478.txt  (40 KB)
  2604.18509.txt  (61 KB)


## A catalog file

The per-paper JSON files are nice for humans but annoying to iterate over in later notebooks. So I'm going to produce one `catalog.json` that lists every paper we have along with its file paths. This is the thing the next notebook (the one where we parse the PDFs) will open first.

In [45]:
catalog = []
for mf in sorted(META_DIR.glob("*.json")):
    m = json.loads(mf.read_text(encoding="utf-8"))
    catalog.append({
        "arxiv_id": m["arxiv_id"],
        "title": m["title"],
        "authors": m["authors"],
        "published": m["published"],
        "txt_path": str(TXT_DIR / f"{m['arxiv_id']}.txt"),
        "meta_path": str(mf),
    })

catalog_path = DATA_DIR / "catalog.json"
catalog_path.write_text(json.dumps(catalog, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"{len(catalog)} entries in {catalog_path}")

26 entries in papers/catalog.json
